# Modern Hopfield Networks
Classical Hopfield networks are elegant but limited: they can only store a small number of binary patterns reliably. Modern Hopfield networks address these shortcomings.

> __Why are these interesting?__ The modern Hopfield energy function generalizes that of the classical network to allow the storage of *continuous* (not just binary) patterns. It also enables the storage of exponentially many (potentially *correlated*) memories and exhibits much faster convergence behavior than classical networks.

The key innovation in modern Hopfield networks is the reformulation of the energy function. [Krotov and Hopfield (2016)](https://arxiv.org/abs/1606.01164) proposed a new energy function of the form:
$$
\begin{align*}
E(\mathbf{s}) &= -\sum_{i=1}^{K}F(\mathbf{m}_{i}^{\top}\mathbf{s}) \\
\end{align*}
$$
where $F$ is a nonlinear function, $\mathbf{m}_i$ is the $i$-th memory, $K$ is the number of memories, and $\mathbf{s}$ is the state of the network. 
The function $F$ maps the similarity (inner product) between the state and memory vectors to a scalar energy value. The choice of $F$ determines the type of memory dynamics and convergence behavior. There are many choices for $F$, but one particularly interesting choice was proposed by [Ramsauer et al. (2020)](https://arxiv.org/abs/2008.02217):
$$
\begin{align*}
E(\mathbf{s}) &= -\texttt{lse}(\beta,\mathbf{X}^{\top}\mathbf{s}) + \frac{1}{2}\mathbf{s}^{\top}\mathbf{s} + \frac{1}{\beta}\log(K)+ \frac{1}{2}M^{2} \\
\end{align*}
$$
where $\mathbf{X}\in\mathbb{R}^{N\times{K}}$ is the matrix of memories, i.e., each memory $\mathbf{m}_{1},\dots,\mathbf{m}_{K}$ consisting of $N$ features is a column of the matrix, $\mathbf{s}$ is the current state of the network, and $\texttt{lse}(\cdot)$ is the log-sum-exp function:
$$
\begin{align*}
\texttt{lse}(\beta,\mathbf{z}) &= \frac{1}{\beta}\log\left(\sum_{i=1}^{K}\exp(\beta\,\mathbf{z}_{i})\right) \\
\end{align*}
$$
and $\beta$ is an inverse temperature parameter that controls the sharpness of the distribution. Finally, $M$ is the largest norm of all the memories, i.e., $M = \max_{i=1,\dots,K}\|\mathbf{m}_{i}\|$. The constants $\frac{1}{\beta}\log(K)$ and $\frac{1}{2}M^2$ ensure the energy remains bounded and comparable across different configurations.

The vector $\mathbf{X}^{\top}\mathbf{s}$ computes the similarity (dot product) between the current state $\mathbf{s}$ and each stored memory, producing a $K$-dimensional vector of similarities. The log-sum-exp function then aggregates these similarities in a smooth, differentiable manner.

### Algorithm: Memory retrieval
The user provides a set of memory vectors $\mathbf{X} = \left\{\mathbf{m}_{1}, \mathbf{m}_{2}, \ldots, \mathbf{m}_{K}\right\}$, where $\mathbf{m}_{i} \in \mathbb{R}^{N}$ is a memory vector of size $N$ and $K$ is the number of memory vectors. Further, the user provides an initial _partial memory_ $\mathbf{s}_{\circ} \in \mathbb{R}^{N}$, which is a vector of size $N$, and specifies the _inverse temperature_ $\beta$ of the system.

__Initialize__ the network with the memory matrix $\mathbf{X}$ and inverse temperature $\beta$. Set the current state $\mathbf{s} \gets \mathbf{s}_{\circ}$, initialize the iteration counter $t \gets 1$, maximum iterations $\texttt{maxiter}$, and set convergence flag $\texttt{converged} \gets \texttt{false}$ and tolerance $\epsilon > 0$.

> **Parameter Guidelines**: Common choices are `maxiter = 1000` and $\epsilon$ = `1e-6`. Modern Hopfield networks typically converge within 10–100 iterations, making `maxiter = 1000` a conservative upper bound. The tolerance $\epsilon$ = `1e-6` provides good precision for most applications while avoiding numerical precision issues.

While not $\texttt{converged}$ and $t \leq \texttt{maxiter}$ __do__:
   1. Compute the _current_ similarity vector $\mathbf{z} = \mathbf{X}^{\top}\mathbf{s}$, where each element $z_i = \mathbf{m}_i^{\top}\mathbf{s}$ represents the similarity between the current state and memory $i$.
   2. Compute the _current_ probability vector $\mathbf{p} = \texttt{softmax}(\beta\cdot\mathbf{z})$ where $\texttt{softmax}(\mathbf{u})_i = \frac{\exp(u_i)}{\sum_{j=1}^{K}\exp(u_j)}$.
   3. Compute the _next_ state vector $\mathbf{s}^{\prime} = \mathbf{X}\mathbf{p}$ using the current probability vector $\mathbf{p}$ and the memory matrix $\mathbf{X}$. This step computes a weighted sum of the memory vectors based on the probabilities.
   4. **Check for convergence**: If $\lVert \mathbf{s}^{\prime} - \mathbf{s}\rVert_{2} \leq \epsilon$, then set $\texttt{converged} \gets \texttt{true}$.
      - **Alternative**: If $\lVert \mathbf{p} - \mathbf{p}_{\text{prev}}\rVert_{1} \leq \epsilon_p$, where $\mathbf{p}_{\text{prev}}$ is the probability vector from the previous iteration, $\epsilon_p$ is the convergence tolerance for probabilities (default: $\epsilon$ = `1e-8`), and $\lVert\star\rVert_{1}$ is the L1-norm, then set $\texttt{converged} \gets \texttt{true}$.
   5. **Update state**: $\mathbf{s} \gets\mathbf{s}^{\prime}$ and increment $t \gets t + 1$.

> **Note**: The softmax function in step 2 is directly related to the log-sum-exp function in the energy formulation. Specifically, the gradient of the LSE with respect to $\mathbf{s}$ yields the softmax-weighted combination of memories used in the update rule.

### Convergence

Modern Hopfield networks have even stronger convergence properties than their classical counterparts, making them highly effective for practical applications.

* **Guaranteed Convergence**: Like classical Hopfield networks, modern variants are **guaranteed to converge** to a fixed point. The energy function serves as a Lyapunov function that decreases monotonically with each update until reaching a minimum.
* **Exponential Convergence Rate**: Modern Hopfield networks exhibit **exponential convergence** to stored memories, dramatically faster than the polynomial convergence of classical networks. The softmax operation creates a "winner-take-all" dynamic that rapidly identifies and converges to the most similar stored pattern.

Let's discuss convergence of modern Hopfield networks in practice.

> **Convergence** 
>
> In practice, modern Hopfield networks converge quickly: in the best case, convergence occurs in 1–5 iterations. However, in the worst case, it may take 100–200 iterations, especially if the initial state is far from any stored memory or if the memories are highly correlated.
>
> **Factors Affecting Convergence**:
> - **Inverse temperature β**: Higher β leads to faster convergence but may reduce the basin of attraction
> - **Memory separation**: Well-separated memories in the feature space converge faster
> - **Initialization quality**: Starting closer to any stored pattern leads to faster convergence

The exponential convergence rate, combined with increased storage capacity and continuous memory representations, makes modern Hopfield networks significantly more practical than classical variants for real-world applications, especially in high-dimensional continuous data scenarios.

___